# Motion energy — example analysis

Demonstrates the `aind-motion-energy` pipeline on a behavior video from an attached data
asset. The notebook:

1. Discovers videos and reports their metadata
2. Computes motion energy on the first 5 000 frames
3. Shows a static summary (timeseries + spatial map)
4. Loads the session NWB file and aligns ME traces to go-cue onset
5. Renders a short example clip around a representative go cue

Run this interactively in Code Ocean JupyterLab with the data asset attached.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from aind_motion_energy import compute_motion_energy, clean_trace, render_motion_energy_video
from aind_motion_energy.io import get_video_info

DATA_DIR   = Path("/root/capsule/data")
SCRATCH_DIR = Path("/root/capsule/scratch")
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_EXTS = {".mp4", ".avi", ".mkv", ".mov", ".mj2"}
videos = sorted(p for p in DATA_DIR.rglob("*") if p.suffix.lower() in VIDEO_EXTS)

for v in videos:
    info = get_video_info(v)
    print(
        f"{v.relative_to(DATA_DIR)}"
        f"  —  {info['n_frames']} frames @ {info['fps']} fps"
        f"  ({info['width']}×{info['height']})"
    )

## 1. Compute motion energy

Change `VIDEO` to select a different camera. `END_FRAME = 5000` keeps runtime short
(~10 s at 500 fps); remove the limit to process the full recording.

In [ ]:
VIDEO     = videos[0]   # change index to select a different camera
END_FRAME = 5000        # set to None for full recording

me, keyframe_mask, avg_map, meta = compute_motion_energy(VIDEO, end_frame=END_FRAME)
me_clean = clean_trace(me, keyframe_mask)

fps = meta["fps"]
t   = np.arange(len(me)) / fps

print(
    f"{VIDEO.stem}: {len(me)} diffs | "
    f"{meta['n_keyframes_masked']} keyframe diffs masked | "
    f"mean ME = {me_clean.mean():.4f}"
)

## 2. Static summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))

axes[0].plot(t, me,       lw=0.4, color="0.78", alpha=0.8)
axes[0].plot(t, me_clean, lw=0.7, color="steelblue")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Motion energy")
axes[0].set_title(VIDEO.stem)
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)

im = axes[1].imshow(avg_map, cmap="hot", aspect="auto")
plt.colorbar(im, ax=axes[1], label="Mean abs diff (per pixel)")
axes[1].set_title("Average motion map (keyframe-excluded)")

fig.tight_layout()
plt.show()

## 3. Load NWB and extract go-cue times

Finds the NWB file in the same data asset as the video, reads the trials table,
and auto-detects the go-cue column (column name varies across sessions).

In [ ]:
import pynwb

# Find NWB in the same data-asset folder as the chosen video
asset_root = next(p for p in DATA_DIR.iterdir() if video.is_relative_to(p)) \
             if False else VIDEO.parents[len(VIDEO.relative_to(DATA_DIR).parts) - 1]
# simpler: scan all NWB files under DATA_DIR
nwb_files = sorted(DATA_DIR.rglob("*.nwb"))
print(f"NWB files found: {[str(f.relative_to(DATA_DIR)) for f in nwb_files]}")

In [ ]:
io  = pynwb.NWBHDF5IO(str(nwb_files[0]), "r")
nwb = io.read()

trials = nwb.trials.to_dataframe()
print("Trial table columns:", trials.columns.tolist())
print(f"{len(trials)} trials")
trials.head()

In [ ]:
# Auto-detect the go-cue column
gocue_col = next(
    (c for c in trials.columns if "gocue" in c.lower() or "go_cue" in c.lower()),
    None,
)
if gocue_col is None:
    raise RuntimeError(f"No go-cue column found. Available: {trials.columns.tolist()}")

gocue_times_s = trials[gocue_col].dropna().values
print(f"Go-cue column: '{gocue_col}'  —  {len(gocue_times_s)} trials")
print(f"First 5 go-cue times (s): {gocue_times_s[:5]}")

## 4. Trial-aligned motion energy

Extracts a window around each go cue and plots the per-trial heatmap and mean ± SEM.

> **Note:** go-cue times are in session time (seconds since session start). If the video
> does not start at t=0 of the session, offset them by subtracting the video's start time
> relative to session start (available in the NWB or the `_video_timestamps.csv` sidecar).

In [ ]:
PRE_S  = 0.5   # seconds before go cue
POST_S = 2.0   # seconds after go cue

pre_fr  = int(PRE_S  * fps)
post_fr = int(POST_S * fps)

# Convert go-cue times to frame indices within the ME array
# (assumes video starts at session t=0; adjust offset below if needed)
SESSION_OFFSET_S = 0.0   # seconds from session start to video frame 0
gocue_frames = ((gocue_times_s - SESSION_OFFSET_S) * fps).astype(int)

aligned = []
for f0 in gocue_frames:
    s, e = f0 - pre_fr, f0 + post_fr
    if 0 <= s and e < len(me_clean):
        aligned.append(me_clean[s:e])

aligned  = np.array(aligned)
trial_t  = np.linspace(-PRE_S, POST_S, aligned.shape[1])
print(f"{len(aligned)} trials with full window within the clip")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

im = axes[0].imshow(
    aligned, aspect="auto",
    extent=[-PRE_S, POST_S, len(aligned), 0],
    cmap="viridis",
)
axes[0].axvline(0, color="white", lw=0.8, ls="--", label="go cue")
axes[0].set_xlabel("Time from go cue (s)")
axes[0].set_ylabel("Trial")
axes[0].set_title("Motion energy per trial")
plt.colorbar(im, ax=axes[0])

mean = aligned.mean(axis=0)
sem  = aligned.std(axis=0) / np.sqrt(len(aligned))
axes[1].fill_between(trial_t, mean - sem, mean + sem, alpha=0.25, color="steelblue")
axes[1].plot(trial_t, mean, color="steelblue", lw=1.2)
axes[1].axvline(0, color="0.4", lw=0.8, ls="--", label="go cue")
axes[1].set_xlabel("Time from go cue (s)")
axes[1].set_ylabel("Motion energy")
axes[1].set_title(f"Mean ± SEM  (n={len(aligned)} trials)")
axes[1].legend(fontsize=8)
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

fig.tight_layout()
plt.show()

## 5. Rendered example clip

Encodes a synced video (footage + scrolling ME plot) for the 4-second window
around the first go cue. Saved to `/root/capsule/scratch/`.

In [ ]:
first_go_fr = gocue_frames[0]
clip_start  = max(0, first_go_fr - int(1.0 * fps))   # 1 s before
clip_end    = first_go_fr + int(3.0 * fps)            # 3 s after

out_path = SCRATCH_DIR / f"{VIDEO.stem}_go_cue_example.mp4"

render_motion_energy_video(
    VIDEO, me_clean,
    fps_source=fps,
    output_path=out_path,
    raw_trace=me,
    start_frame=clip_start,
    end_frame=clip_end,
    window_seconds=2.0,
    out_fps=60.0,
)
print(f"Saved: {out_path}")